# 04 - H5: Aufmerksamkeits-Volumen-Diskrepanz

**Kernfrage.** Spiegelt die parlamentarische Aufmerksamkeit auf
digitale Themen, gemessen über Bundestagsreden, die tatsächliche
Mittelverteilung im Digitalhaushalt wider? Oder gibt es eine
systematische Diskrepanz zwischen dem, was die Politik bespricht, und
dem, was sie finanziert?

**Essay-Bezug.** Akt 2b (Aufmerksamkeit vs. Volumen) und Limitations-
Sektion. Verschneidung des ZEW-Digitalhaushalts mit dem CPP-BT-Korpus
der Bundestagsreden (Sean Fobbe, Zenodo, Version 2026-01-17).

## Datenbasis und methodische Anmerkung

**Primärdaten:** ZEW-Digitalhaushalt, geladen über `src.load.load()`.

**Verschneidungs-Daten:** **CPP-BT** Corpus der Plenarprotokolle des
Deutschen Bundestages (Fobbe 2026), Version 2026-01-17, Public Domain
(CC0). Enthält Einzelreden ab der 18. Wahlperiode mit Metadaten zu
Fraktion, Name, Datum und Volltext. Quelle:
https://doi.org/10.5281/zenodo.4542661

**Bewusst nicht verwendet:** Der ursprünglich als zweite Verschneidung
geplante Drucksachen-Korpus **CDRS-BT** wurde verworfen, die einzige
öffentlich verfügbare Version (2021-04-02) deckt nur die Wahlperioden
1–18 (bis 2017) ab und damit nicht unsere Analyse-Jahre 2019, 2021,
2023 und 2024. Diese Limitation wird im Essay-Methodenanhang explizit
benannt.

In [1]:
import sys
import re
from pathlib import Path
sys.path.insert(0, '..')  # repo root, relativ zum notebooks/-Verzeichnis

import pandas as pd
from src.load import load
from src import OUTPUTS_DIR, REPO_ROOT
from src.reden_mapping import (
    DIGITAL_BEGRIFFE, RESSORT_BEGRIFFE, RESSORT_TO_EP, EP_TO_RESSORT,
)

## Pfad zur Reden-Datei

Erwartung: Der CPP-BT-Reden-Korpus liegt unter `data/external/`. Die
Datei ist ca. 118 MB als ZIP und wird direkt von pandas gelesen, ohne
vorher entpackt zu werden. Falls die Datei fehlt, gibt das Notebook
eine klare Fehlermeldung und beendet sich.

In [2]:
EXTERNAL = REPO_ROOT / 'data' / 'external'
CPP_PATH = EXTERNAL / 'CPP-BT_2026-01-17_DE_CSV_Reden_Gesamt.zip'

if not CPP_PATH.exists():
    print(f"STOP: Reden-Korpus nicht gefunden unter {CPP_PATH}")
    print(f"\nLaden von https://zenodo.org/records/18177196")
    print("Konkret die Datei: CPP-BT_2026-01-17_DE_CSV_Reden_Gesamt.zip")
    print(f"\nAblage unter: {EXTERNAL}")
    print("\nVerfuegbare Dateien im Verzeichnis:")
    if EXTERNAL.exists():
        for f in EXTERNAL.iterdir():
            if f.is_file():
                print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
    sys.exit(0)

print(f"OK: {CPP_PATH.name} ({CPP_PATH.stat().st_size / 1e6:.1f} MB)")

OK: CPP-BT_2026-01-17_DE_CSV_Reden_Gesamt.zip (117.8 MB)


## Strukturpruefung

Bevor wir den vollen Datensatz laden, prüfen wir die Spaltenstruktur
an den ersten fünf Zeilen. Die Spaltennamen der CPP-BT-Variablen
können sich zwischen Versionen ändern (z. B. wurde `nummer_original`
in der 2026-Version zu `protokoll_nr` umbenannt). Diese Prüfung deckt
Drift früh auf.

In [3]:
print(f"=== Strukturpruefung CPP-BT ===")
sample = pd.read_csv(CPP_PATH, compression='zip', nrows=5, encoding='utf-8')
print(f"Spalten ({len(sample.columns)}): {sample.columns.tolist()}")
print("\nErste Zeile (gekuerzt):")
for col in sample.columns:
    val = str(sample.iloc[0][col]) if len(sample) > 0 else ''
    preview = val[:120] + ('...' if len(val) > 120 else '')
    print(f"  {col:<30}: {preview}")

=== Strukturpruefung CPP-BT ===
Spalten (35): ['doc_id', 'wahlperiode', 'sitzung_nr', 'sitzung_datum', 'sitzung_jahr', 'sitzung_start', 'sitzung_ende', 'sitzung_naechste', 'dokumentart', 'berichtart', 'protokoll_nr', 'protokoll_seite', 'sitzungstitel', 'sitzungstitel_zusatz', 'rede_id', 'rede_text', 'rede_zitate', 'rede_kommentare', 'redner_id', 'redner_titel', 'redner_vorname', 'redner_nachname', 'redner_namenszusatz', 'redner_ortszusatz', 'redner_fraktion', 'redner_rolle_kurz', 'redner_rolle_lang', 'zeichen', 'tokens', 'typen', 'saetze', 'version', 'doi_concept', 'doi_version', 'lizenz']

Erste Zeile (gekuerzt):
  doc_id                        : 18001.xml-0001
  wahlperiode                   : 18
  sitzung_nr                    : 1
  sitzung_datum                 : 2013-10-22
  sitzung_jahr                  : 2013
  sitzung_start                 : 11:00
  sitzung_ende                  : 15:01
  sitzung_naechste              : nan
  dokumentart                   : Plenarprotokoll
  be

## Spalten-Mapping (eventuell nach Strukturpruefung anzupassen)

Die folgenden Konstanten verweisen auf die Spalten des Reden-Korpus,
die wir für die Aufmerksamkeitsanalyse brauchen. Falls die
Strukturprüfung oben zeigt, dass die Namen abweichen, hier anpassen.

In [4]:
REDE_DATUM_COL = 'sitzung_datum'    # ggf. anpassen
REDE_TEXT_COL = 'rede_text'         # ggf. anpassen
REDE_FRAKTION_COL = 'redner_fraktion'  # ggf. anpassen

## Reden laden, gefiltert auf 2019-2024

Wir analysieren nur die Jahre, die auch im ZEW-Datensatz enthalten
sind: 2019, 2021, 2023, 2024. Das spart Speicher und macht den
Vergleich sauber.

In [5]:
print("\n=== Reden laden, gefiltert auf 2019-2024 ===")
pp_raw = pd.read_csv(CPP_PATH, compression='zip', encoding='utf-8')
print(f"  Gesamt: {len(pp_raw):,} Reden")

pp_raw[REDE_DATUM_COL] = pd.to_datetime(pp_raw[REDE_DATUM_COL], errors='coerce')
pp = pp_raw[pp_raw[REDE_DATUM_COL].dt.year.isin([2019, 2021, 2023, 2024])].copy()
pp['jahr'] = pp[REDE_DATUM_COL].dt.year
print(f"  Im Zeitraum 2019-2024: {len(pp):,}")
print(f"  Pro Jahr: {pp.groupby('jahr').size().to_dict()}")


=== Reden laden, gefiltert auf 2019-2024 ===
  Gesamt: 78,188 Reden
  Im Zeitraum 2019-2024: 30,458
  Pro Jahr: {2019: 7511, 2021: 5580, 2023: 9256, 2024: 8111}


/var/folders/xm/gs28jq6d06v82tq2zhvmw96r0000gn/T/ipykernel_93479/1343539449.py:2: DtypeWarning: Columns (0: rede_zitate, 1: redner_id, 2: redner_namenszusatz) have mixed types. Specify dtype option on import or set low_memory=False.
  pp_raw = pd.read_csv(CPP_PATH, compression='zip', encoding='utf-8')


## Ressort- und Digital-Treffer zählen

Methodik: Pro Rede prüfen wir, ob sie sowohl ein Ressort-typisches
Schlagwort als auch ein Digital-Schlagwort enthält. Eine Rede zählt
nur dann als Digital-Aufmerksamkeit für ein Ressort, wenn beide
Bedingungen erfüllt sind.

Das Vokabular liegt in `src/reden_mapping.py`. `DIGITAL_BEGRIFFE` sind
30+ digitale Schlagwörter, abgeleitet aus den ZEW-Tags.
`RESSORT_BEGRIFFE` ordnet jedem Ressort 5–10 ressortspezifische
Begriffe zu.

In [6]:
def zaehle_treffer(text_series, begriffe):
    """Markiert Texte, die mindestens einen der Begriffe enthalten.

    Wortgrenzen verhindern, dass etwa 'kit' in 'aktivität' fälschlich
    als KI-Treffer gilt.
    """
    pattern = '|'.join(r'\b' + re.escape(b) + r'\b' for b in begriffe)
    return text_series.fillna('').str.contains(pattern, case=False, regex=True)


mask_digital = zaehle_treffer(pp[REDE_TEXT_COL], DIGITAL_BEGRIFFE)
print(f"\nReden mit mindestens einem Digital-Begriff: {mask_digital.sum():,} "
      f"({mask_digital.mean()*100:.1f} % aller Reden 2019-2024)")

results = []
for ressort, begriffe in RESSORT_BEGRIFFE.items():
    mask_ressort = zaehle_treffer(pp[REDE_TEXT_COL], begriffe)
    mask_kombi = mask_ressort & mask_digital
    sub = pp[mask_kombi]
    for jahr in [2019, 2021, 2023, 2024]:
        n = (sub['jahr'] == jahr).sum()
        results.append({'ressort': ressort, 'jahr': jahr, 'reden_digital': n})

df_reden = pd.DataFrame(results)
print("\n=== Digital-Reden je Ressort und Jahr ===")
print(df_reden.pivot(index='ressort', columns='jahr',
                    values='reden_digital').fillna(0))


Reden mit mindestens einem Digital-Begriff: 3,736 (12.3 % aller Reden 2019-2024)

=== Digital-Reden je Ressort und Jahr ===
jahr     2019  2021  2023  2024
ressort                        
BMBF       84    44    53    66
BMDV       64    17    32    24
BMF        39    12    18    18
BMG        21    20    30    16
BMI        64    47    84    83
BMVg       33    17    28    35
BMWK       20    10    20    13
EP60        0     0     0     0


## Normierung auf Reden-Gesamt pro Jahr

Absolute Trefferzahlen sind irreführend, weil sich die Sitzungszahl
über die Jahre ändern kann (Wahljahre haben weniger Sitzungen). Wir
normieren auf die Gesamtzahl aller Reden pro Jahr, der „Aufmerksamkeits-
Anteil" zeigt, welcher Anteil der gesamten parlamentarischen Redezeit
auf das Ressort-Digital-Thema entfällt.

In [7]:
pp_pro_jahr = pp.groupby('jahr').size().to_dict()
df_reden['reden_anteil_promille'] = df_reden.apply(
    lambda r: r['reden_digital'] / pp_pro_jahr[r['jahr']] * 1000, axis=1
)

print("Aufmerksamkeits-Anteil in Promille (je Ressort und Jahr):")
print(df_reden.pivot(index='ressort', columns='jahr',
                    values='reden_anteil_promille').round(2).fillna(0))

Aufmerksamkeits-Anteil in Promille (je Ressort und Jahr):
jahr      2019  2021  2023   2024
ressort                          
BMBF     11.18  7.89  5.73   8.14
BMDV      8.52  3.05  3.46   2.96
BMF       5.19  2.15  1.94   2.22
BMG       2.80  3.58  3.24   1.97
BMI       8.52  8.42  9.08  10.23
BMVg      4.39  3.05  3.03   4.32
BMWK      2.66  1.79  2.16   1.60
EP60      0.00  0.00  0.00   0.00


## Join mit ZEW-Daten

Pro Ressort und Jahr addieren wir das digitale Soll-Volumen
(`digi_soll_eng`) und führen es mit den Reden-Anteilen zusammen.

In [8]:
print("\n=== Join mit ZEW-Digital-Soll ===")
zew = load()
zew_agg = (
    zew.groupby(['einzelplan', 'jahr'])['digi_soll_eng']
    .sum().div(1e6).reset_index()
    .rename(columns={'digi_soll_eng': 'mrd_eur'})
)
zew_agg['ressort'] = zew_agg['einzelplan'].map(EP_TO_RESSORT)
zew_agg = zew_agg.dropna(subset=['ressort'])

panel = zew_agg.merge(df_reden, on=['ressort', 'jahr'], how='left')
panel = panel[['ressort', 'einzelplan', 'jahr', 'mrd_eur',
               'reden_digital', 'reden_anteil_promille']]
print(panel.round(2).to_string(index=False))


=== Join mit ZEW-Digital-Soll ===
ressort  einzelplan  jahr  mrd_eur  reden_digital  reden_anteil_promille
    BMI           6  2019     1.90             64                   8.52
    BMI           6  2021     3.49             47                   8.42
    BMI           6  2023     2.39             84                   9.08
    BMI           6  2024     1.98             83                  10.23
    BMF           8  2019     0.78             39                   5.19
    BMF           8  2021     1.49             12                   2.15
    BMF           8  2023     1.89             18                   1.94
    BMF           8  2024     2.00             18                   2.22
   BMWK           9  2019     0.90             20                   2.66
   BMWK           9  2021     1.33             10                   1.79
   BMWK           9  2023     2.37             20                   2.16
   BMWK           9  2024     1.38             13                   1.60
   BMDV         

## Diskrepanz-Indikator

Pro Jahr ranken wir die Ressorts nach (a) ihrem Anteil am
Digital-Soll und (b) ihrem Aufmerksamkeits-Anteil. Die Differenz der
Ränge ist der Diskrepanz-Indikator:

- **Positiv**: Ressort bekommt mehr parlamentarische Aufmerksamkeit
  als sein Volumen rechtfertigen würde (Buzz-Ressort).
- **Negativ**: Ressort bekommt weniger Aufmerksamkeit als sein Volumen
  rechtfertigen würde (stilles Schwergewicht).
- **Null oder nahe Null**: Match, Aufmerksamkeit und Volumen im
  Gleichgewicht.

In [9]:
print("\n=== Diskrepanz-Indikator ===")
for jahr in [2019, 2024]:
    sub = panel[panel['jahr'] == jahr].copy()
    sub['rang_volumen'] = sub['mrd_eur'].rank(ascending=False)
    sub['rang_aufmerksamkeit'] = sub['reden_anteil_promille'].rank(ascending=False)
    sub['diskrepanz'] = sub['rang_aufmerksamkeit'] - sub['rang_volumen']
    sub = sub.sort_values('diskrepanz')
    print(f"\n--- {jahr} (sortiert: stille Schwergewichte oben, Buzz-Ressorts unten) ---")
    print(sub[['ressort', 'mrd_eur', 'reden_anteil_promille',
               'rang_volumen', 'rang_aufmerksamkeit',
               'diskrepanz']].round(2).to_string(index=False))


=== Diskrepanz-Indikator ===

--- 2019 (sortiert: stille Schwergewichte oben, Buzz-Ressorts unten) ---
ressort  mrd_eur  reden_anteil_promille  rang_volumen  rang_aufmerksamkeit  diskrepanz
    BMF     0.78                   5.19           6.0                  4.0        -2.0
    BMG     0.03                   2.80           8.0                  6.0        -2.0
   BMBF     1.59                  11.18           3.0                  1.0        -2.0
   BMDV     0.93                   8.52           4.0                  2.5        -1.5
    BMI     1.90                   8.52           2.0                  2.5         0.5
   EP60     0.05                   0.00           7.0                  8.0         1.0
   BMWK     0.90                   2.66           5.0                  7.0         2.0
   BMVg     1.97                   4.39           1.0                  5.0         4.0

--- 2024 (sortiert: stille Schwergewichte oben, Buzz-Ressorts unten) ---
ressort  mrd_eur  reden_anteil_promille

## Export der Panel-Daten

Das Panel wird als CSV exportiert, damit der nachfolgende Hero-Chart
auf den ausgewerteten Zahlen aufsetzen kann.

In [10]:
out = OUTPUTS_DIR / 'h5_panel.csv'
out.parent.mkdir(exist_ok=True)
panel.to_csv(out, index=False)
print(f"\nPanel-Daten exportiert nach: {out}")


Panel-Daten exportiert nach: /Users/moe/Documents/GitHub/agora_challenge/outputs/h5_panel.csv


## Schluss

**Kernbefund.** Die Daten zeigen eine strukturelle Aufmerksamkeits-Volumen-Diskrepanz,
die der ursprünglichen Erwartung teils widerspricht.

Das markanteste **stille Schwergewicht** 2024 ist nicht das Verteidigungsministerium
(BMVg), sondern das **BMDV**: Es hält mit 4,12 Mrd. € das größte Digital-Soll aller
Ressorts (Volumen-Rang 1), erhält parlamentarisch aber nur Rang 4 in der digitalen
Aufmerksamkeit (2,96 ‰). Diskrepanz-Index: +3. Die Breitband- und ETCS-
Transfer-Investitionen, die den BMDV-Etat dominieren, erzeugen im parlamentarischen
Digitalraum deutlich weniger Debattenraum als ihr Volumen erwarten ließe.

Das ausgeprägteste **Buzz-Ressort** 2024 ist das **BMI**: Rang 5 im Volumen
(1,98 Mrd. €), aber Rang 1 in der Aufmerksamkeit (10,23 ‰). Diskrepanz-Index: −4.
Cybersicherheit, BSI, Digitalfunk und digitale Verwaltungsthemen des BMI ziehen
parlamentarische Debatten stärker an, als es das Mittelvolumen erwarten ließe.

**BMBF** und **BMVg** zeigen 2024 ausgeglichene Match-Profile (Diskrepanz 0): ihre
parlamentarische Sichtbarkeit entspricht ihrem Anteil am Digitalhaushalt.

**Verbindung zu Akt 1.** Dieses Ergebnis ergänzt die Polyzentrik-Diagnose um eine
zweite Achse: Der Digitalhaushalt ist nicht nur in seinen Mitteln (H1), sondern auch
in seiner parlamentarischen Aufmerksamkeit mehrdimensional strukturiert. Die
Polyzentrik der Debatte ist nicht deckungsgleich mit der Polyzentrik der Mittel.
BMDV regiert das Geld, BMI regiert die Schlagzeilen.

**Methodische Einschränkung.** Das Ressort-Mapping über Schlagwörter ist eine
approximative Methode; Reden besprechen Themen, nicht Haushaltspositionen. Der
fehlende Drucksachen-Korpus (CDRS-BT, nur bis 2017 verfügbar) ist eine weitere
Limitation. Die Ergebnisse sind als Exploration zu verstehen, nicht als
validierten abschließenden Befund. Eine zweite Verschneidungsschiene über
Drucksachen würde die Befunde in einer Folgeversion validieren.